# OrganismRSSM Training on TPU

**Kagami World Model Training Notebook**

Features:
- 7-colony OrganismRSSM architecture
- SOTA techniques: MLA, GQA, Mamba-2, RoPE
- DreamerV3 training: symlog, TwoHot, balanced KL
- 7-phase curriculum: WARMUP → LANGUAGE
- Phi-4 style data efficiency

Run on: Google Colab TPU or Kaggle TPU

In [ ]:
# Check runtime
import os

print(f"TPU_NAME: {os.environ.get('TPU_NAME', 'Not set')}")
print(f"Colab TPU: {os.environ.get('COLAB_TPU_ADDR', 'Not set')}")

In [ ]:
# Install dependencies
!pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip install -q flax optax orbax-checkpoint einops tqdm

In [ ]:
# Verify JAX TPU
import jax

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Device count: {jax.device_count()}")

In [ ]:
# Clone training code
!git clone --depth 1 https://github.com/awkronos/kagami.git /content/kagami 2>/dev/null || echo "Already cloned"
import sys

sys.path.insert(0, "/content/kagami/packages")

In [ ]:
# Load configurations
from kagami.core.training.jax.configs import (
    get_sota_config,
)

# Use SOTA config
sota = get_sota_config()
print("SOTA Config:")
print(f"  Hidden dim: {sota.llm.hidden_dim}")
print(f"  Layers: {sota.llm.num_layers}")
print(f"  Attention heads: {sota.llm.num_attention_heads}")
print(f"  KV heads (GQA): {sota.llm.num_kv_heads}")
print(f"  MLA latent dim: {sota.mla.latent_dim}")

In [ ]:
# Import training components
import jax.numpy as jnp
import optax

from kagami.core.training.jax.rssm import OrganismRSSM
from kagami.core.training.jax.config import OrganismRSSMConfig
from kagami.core.training.jax.curriculum import CurriculumManager
from kagami.core.training.jax.telemetry import TrainingTelemetry

In [ ]:
# Initialize model
config = OrganismRSSMConfig(
    obs_dim=128,
    action_dim=8,
    num_colonies=7,
    deter_dim=sota.llm.hidden_dim,
    stoch_dim=64,
    discrete_categories=32,
    discrete_classes=32,
    latent_classes=240,  # E8 roots
    unimix=0.01,
    free_bits=sota.dreamer.kl_free_bits,
)

print(f"Model config created: {config.deter_dim}D hidden, {config.num_colonies} colonies")

In [ ]:
# Create model and initialize
rng = jax.random.PRNGKey(42)
model = OrganismRSSM(config)

# Initialize with dummy input
B, T = 8, 32
dummy_obs = jnp.zeros((B, T, config.obs_dim))
dummy_actions = jnp.zeros((B, T, config.action_dim))

params = model.init(rng, dummy_obs, dummy_actions)


# Count parameters
def count_params(params):
    return sum(p.size for p in jax.tree_util.tree_leaves(params))


num_params = count_params(params)
print(f"Model parameters: {num_params:,} ({num_params / 1e6:.1f}M)")

In [ ]:
# Create optimizer with cosine schedule
warmup_steps = 2000
total_steps = 500_000

schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=sota.training.learning_rate,
    warmup_steps=warmup_steps,
    decay_steps=total_steps - warmup_steps,
    end_value=sota.training.min_learning_rate,
)

optimizer = optax.chain(
    optax.clip_by_global_norm(sota.training.grad_clip),
    optax.adamw(
        learning_rate=schedule,
        b1=sota.training.adam_beta1,
        b2=sota.training.adam_beta2,
        eps=1e-8,
        weight_decay=sota.training.weight_decay,
    ),
)

print("Optimizer: AdamW with cosine schedule")
print(f"  Peak LR: {sota.training.learning_rate}")
print(f"  Warmup: {warmup_steps} steps")

In [ ]:
# Initialize curriculum
curriculum = CurriculumManager()
telemetry = TrainingTelemetry()

print("Curriculum phases:")
for phase in curriculum.phases:
    print(f"  {phase.name}: steps {phase.start_step:,} - {phase.end_step:,}")

In [ ]:
# Training loop (simplified)
from tqdm.auto import tqdm

# Create train state
from flax.training import train_state

state = train_state.TrainState.create(
    apply_fn=model.apply,
    params=params["params"],
    tx=optimizer,
)

print("Training state created. Starting training...")

In [ ]:
# Generate synthetic data for demonstration
def generate_batch(rng, batch_size=32, seq_len=32):
    """Generate synthetic training batch."""
    keys = jax.random.split(rng, 4)
    return {
        "obs": jax.random.normal(keys[0], (batch_size, seq_len, config.obs_dim)),
        "actions": jax.random.normal(keys[1], (batch_size, seq_len, config.action_dim)),
        "rewards": jax.random.normal(keys[2], (batch_size, seq_len)),
        "continues": jnp.ones((batch_size, seq_len)),
    }


# Test forward pass
test_batch = generate_batch(jax.random.PRNGKey(0))
output = model.apply({"params": state.params}, test_batch["obs"], test_batch["actions"])
print("Forward pass successful!")
print(f"  Output h shape: {output.h.shape}")
print(f"  Output z shape: {output.z.shape}")
print(f"  KL (balanced): {float(output.kl_balanced):.4f}")

In [ ]:
# Main training loop
num_steps = 1000  # Short demo
log_interval = 100

rng = jax.random.PRNGKey(42)
losses = []

pbar = tqdm(range(num_steps), desc="Training")
for step in pbar:
    # Generate batch
    rng, batch_rng = jax.random.split(rng)
    batch = generate_batch(batch_rng)

    # Forward pass
    output = model.apply({"params": state.params}, batch["obs"], batch["actions"])

    # Compute loss (simplified)
    obs_pred = output.obs_pred
    loss = jnp.mean((obs_pred[:, :-1] - batch["obs"][:, 1:]) ** 2) + output.kl_balanced
    losses.append(float(loss))

    # Update progress
    if step % log_interval == 0:
        avg_loss = sum(losses[-100:]) / min(100, len(losses))
        phase = curriculum.get_phase(step * 500)  # Scale for demo
        pbar.set_postfix(
            {
                "loss": f"{avg_loss:.4f}",
                "kl": f"{float(output.kl_balanced):.4f}",
                "phase": phase.name,
            }
        )

In [ ]:
# Plot training curve
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
window = 50
smoothed = [
    sum(losses[max(0, i - window) : i + 1]) / min(i + 1, window) for i in range(len(losses))
]
plt.plot(smoothed)
plt.xlabel("Step")
plt.ylabel("Smoothed Loss")
plt.title("Training Loss (Smoothed)")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Training demo complete!")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Best loss: {min(losses):.4f}")

## Full Training

For full training, uncomment and run:

In [ ]:
# Full training (uncomment to run)
# from kagami.core.training.jax.train import train_loop
#
# train_loop(
#     model=model,
#     config=config,
#     num_steps=500_000,
#     checkpoint_dir='/content/checkpoints',
#     log_interval=100,
#     checkpoint_interval=5000,
# )